In [86]:
"""
Monetary Policy Transmission: VAR and DSGE Analysis
Part 1 — Output Gap Construction
Author: Aryan Pandey
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.tsa.filters.hp_filter import hpfilter
import warnings
warnings.filterwarnings('ignore')
import openpyxl
from statsmodels.tsa.vector_ar.var_model import VAR

import statsmodels
print(statsmodels.__version__)


0.14.6


In [220]:
"""
Part 1 (continued) — Output Gap Construction
Four output gap measures:
  1. MN Labour Gap          — 0.7 × (n_t − n̄), n̄ = sample mean of n_t
  2. Log-linear Trend Gap   — log(GDP) − piecewise trend (full sample)
  3. HP Filter              — HP cycle from log(GDP), λ=1600
  4. Band-Pass Filter       — Baxter-King, 6–32 quarters, K=12

Figure 1: All four gaps overlaid, full sample
Figure 2: Sample sensitivity — MN and Log-linear only
          Three windows varying START date (following MN Figure 1.1 logic):
          Full sample (from 1950), From 1980, From 1966
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.tsa.filters.hp_filter import hpfilter
import warnings
import openpyxl
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD DATA
# =============================================================================
wb = openpyxl.load_workbook('EC924_2025_Asiifnment_DataDistribution.xlsx')
ws = wb['Quarterly']
rows = list(ws.iter_rows(values_only=True))
df = pd.DataFrame(rows[1:], columns=rows[0])
df = df.dropna(subset=['observation_date'])
df['date'] = pd.to_datetime(df['observation_date'])
df = df.set_index('date')

ws_m = wb['Monthly']
rows_m = list(ws_m.iter_rows(values_only=True))
dfm = pd.DataFrame(rows_m[1:], columns=rows_m[0])
dfm = dfm.dropna(subset=['observation_date'])
dfm['date'] = pd.to_datetime(dfm['observation_date'])
dfm = dfm.set_index('date')

clf_quarterly = dfm['CLF16OV'].resample('QS').mean()
df['clf'] = clf_quarterly

# =============================================================================
# 2. CONSTRUCT LOG VARIABLES
# =============================================================================
df['lngdp'] = np.log(df['GDPC1'])
df['n_t']   = np.log(df['HOANBS']) - np.log(df['clf'])

# =============================================================================
# 3. TREND ESTIMATION — LOG-LINEAR GAP ONLY, FULL AVAILABLE SAMPLE
# MN gap uses sample mean of n_t — no trend needed
# Log-linear gap uses piecewise trend per assignment instruction
# Estimated on full available sample to avoid explosive extrapolation
# =============================================================================
est = df[['lngdp', 'n_t']].dropna().copy()
est['t']        = np.arange(len(est))
est['d75']      = (est.index >= '1975-01-01').astype(float)
est['t_post75'] = est['t'] * est['d75']

X_gdp       = add_constant(est[['t', 'd75', 't_post75']])
model_trend = OLS(est['lngdp'], X_gdp).fit()

print(f"Log-linear trend estimated on full sample: "
      f"{est.index[0].date()} to {est.index[-1].date()}, N={len(est)}")
print(f"  R²={model_trend.rsquared:.4f}  "
      f"pre-break slope={model_trend.params['t']:.6f}  "
      f"post-break slope change={model_trend.params['t_post75']:.6f}")

# =============================================================================
# 4. COMPUTE ALL FOUR GAPS — FULL SAMPLE
# =============================================================================
full = df[['lngdp', 'n_t']].dropna().copy()

full['t']        = np.arange(len(full))
full['d75']      = (full.index >= '1975-01-01').astype(float)
full['t_post75'] = full['t'] * full['d75']

Xf = add_constant(full[['t', 'd75', 't_post75']])
full['gdp_trend'] = Xf @ model_trend.params

# Gap 1: MN Labour Gap
# ỹ_t = 0.7 × (n_t − n̄) where n̄ = sample mean of n_t over 1950–2000
# MN argue under inelastic labour supply n̄_t is approximately constant
# Use 1950–2000 to match MN's original estimation window
n_bar_q1     = full['n_t'].dropna().loc['1950-01-01':'2000-10-01'].mean()
full['gap_MN'] = 0.7 * (full['n_t'] - n_bar_q1)

print(f"\nMN gap: n̄ = {n_bar_q1:.4f} (mean of n_t over 1950–2000)")

# Gap 2: Log-linear Trend Gap
# Direct application of assignment instruction to log GDP
# Piecewise trend with break at 1975 Q1
full['gap_trend'] = full['lngdp'] - full['gdp_trend']

# Gap 3: HP Filter (λ=1600, quarterly standard)
lngdp_series = full['lngdp'].dropna()
cycle_hp, _ = hpfilter(lngdp_series, lamb=1600)
full['gap_hp'] = np.nan
full.loc[lngdp_series.index, 'gap_hp'] = cycle_hp

# Gap 4: Baxter-King Band-Pass Filter (6–32 quarters, K=12)
def baxter_king(series, low=6, high=32, K=12):
    n = len(series)
    omega_l = 2 * np.pi / high
    omega_h = 2 * np.pi / low
    weights = np.zeros(2*K+1)
    for k in range(-K, K+1):
        if k == 0:
            weights[k+K] = (omega_h - omega_l) / np.pi
        else:
            weights[k+K] = (np.sin(k*omega_h) - np.sin(k*omega_l)) / (np.pi * k)
    weights -= weights.sum() / len(weights)
    result = np.full(n, np.nan)
    vals = series.values
    for t in range(K, n-K):
        result[t] = np.dot(weights, vals[t-K:t+K+1])
    return pd.Series(result, index=series.index)

full['gap_bp'] = baxter_king(full['lngdp'].dropna(), low=6, high=32, K=12)

print(f"\nFull sample: {full.index[0].date()} to {full.index[-1].date()}")
print(f"Band-pass valid: {full['gap_bp'].dropna().index[0].date()} "
      f"to {full['gap_bp'].dropna().index[-1].date()} "
      f"(loses 12 qtrs each end)")

# =============================================================================
# 5. SAMPLE SENSITIVITY — MN AND LOG-LINEAR ONLY
# MN gap: sample mean of n_t over each window (shifts mean, not trend)
# Log-linear gap: piecewise trend fitted over each window
# Three windows vary START date following MN Figure 1.1 logic
# =============================================================================
df_all = df[['lngdp', 'n_t']].dropna()

def build_gaps_from(df_all, est_start):
    """
    MN gap: 0.7 × (n_t − n̄) where n̄ = mean of n_t over this window.
    Log-linear gap: residual from piecewise trend fitted over this window.
    If est_start >= 1975 Q1, structural break regressors dropped to
    avoid perfect collinearity.
    """
    samp = df_all.loc[est_start:].copy()
    samp['t']        = np.arange(len(samp))
    samp['d75']      = (samp.index >= '1975-01-01').astype(float)
    samp['t_post75'] = samp['t'] * samp['d75']

    # MN gap: sample mean over this window
    n_bar = samp['n_t'].mean()
    gap1  = 0.7 * (samp['n_t'] - n_bar)

    # Log-linear gap: piecewise trend over this window
    if pd.Timestamp(est_start) >= pd.Timestamp('1975-01-01'):
        X = add_constant(samp[['t']])
    else:
        X = add_constant(samp[['t', 'd75', 't_post75']])

    m_gdp = OLS(samp['lngdp'], X).fit()
    gap2  = m_gdp.resid

    return gap1, gap2

# Dict order controls plot order — From 1966 plotted last = drawn on top
windows = {
    'Full sample (from 1950)': '1950-01-01',
    'From 1980':               '1980-01-01',
    'From 1966':               '1966-01-01',
}

sensitivity = {}
for label, start in windows.items():
    g1, g2 = build_gaps_from(df_all, start)
    sensitivity[label] = {'gap1': g1, 'gap2': g2}
    print(f"  {label}: {g1.index[0].date()} to {g1.index[-1].date()}, "
          f"N={len(g1)}")

# =============================================================================
# 6. SUMMARY STATISTICS
# =============================================================================
s = full.loc['1966-01-01':'2000-10-01']
print("\n=== Summary statistics 1966 Q1–2000 Q4 ===")
print(s[['gap_MN', 'gap_trend', 'gap_hp', 'gap_bp']].describe().round(4))
print("\nCorrelation matrix:")
print(s[['gap_MN', 'gap_trend', 'gap_hp', 'gap_bp']].corr().round(3))

# =============================================================================
# 7. SHARED PLOT HELPERS
# =============================================================================
RECESSIONS = [
    ('1953-07-01', '1954-05-01'), ('1957-08-01', '1958-04-01'),
    ('1960-04-01', '1961-02-01'), ('1969-12-01', '1970-11-01'),
    ('1973-11-01', '1975-03-01'), ('1980-01-01', '1980-07-01'),
    ('1981-07-01', '1982-11-01'), ('1990-07-01', '1991-03-01'),
    ('2001-03-01', '2001-11-01'), ('2007-12-01', '2009-06-01'),
    ('2020-02-01', '2020-04-01'),
]
BREAK = pd.Timestamp('1975-01-01')
GRAY  = '#888780'
AMBER = '#BA7517'

COL_MN    = '#2C2C2A'    # near-black — MN (theoretically motivated)
COL_TREND = '#C04828'    # red        — linear detrend
COL_HP    = '#558B2F'    # olive green — HP filter
COL_BP    = '#1565C0'    # deep blue  — band-pass

LINE_STYLES = {
    'Full sample (from 1950)': ('-',  1.4, '#2C2C2A', 3),
    'From 1980':               (':',  1.6, '#C04828', 4),
    'From 1966':               (':',  2.2, '#1565C0', 5),
}

def style_ax(ax):
    ax.set_facecolor('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#D3D1C7')
    ax.spines['bottom'].set_color('#D3D1C7')
    ax.tick_params(colors='black', labelsize=9)
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.axhline(0, color=GRAY, lw=0.8, ls='--', alpha=0.7)
    ax.axvline(BREAK, color=AMBER, lw=1.2, ls=':', alpha=0.85, zorder=4)
    ax.margins(x=0)
    ax.grid(axis='y', color=GRAY, linestyle=':', linewidth=0.5, alpha=0.5)

def shade_recessions(ax, idx):
    for rs, re in RECESSIONS:
        s = max(pd.Timestamp(rs), idx[0])
        e = min(pd.Timestamp(re), idx[-1])
        if s < e:
            ax.axvspan(s, e, alpha=0.09, color=GRAY, lw=0)

# =============================================================================
# 8. FIGURE 1 — All four gaps overlaid, full sample
# =============================================================================
fig1, ax1 = plt.subplots(figsize=(13, 5), dpi=130)
fig1.patch.set_facecolor('white')
style_ax(ax1)
shade_recessions(ax1, full.index)

ax1.plot(full.index, full['gap_trend'] * 100,
         color=COL_TREND, lw=1.5, ls='-',
         label='Linear detrend (piecewise, 1975 Q1 break)',
         zorder=3)
ax1.plot(full.index, full['gap_hp'] * 100,
         color=COL_HP, lw=1.5, ls='--',
         label='HP filter (λ=1600)',
         zorder=3)
ax1.plot(full.index, full['gap_bp'] * 100,
         color=COL_BP, lw=1.5, ls=':',
         label='Band-pass filter (6–32 qtrs, K=12)',
         zorder=3)
ax1.plot(full.index, full['gap_MN'] * 100,
         color=COL_MN, lw=2.0, ls='-',
         label='Labour gap — MN (theoretically motivated)',
         zorder=4)

ax1.set_ylabel('Output gap (%)', fontsize=11, fontweight='bold', color='black')
ax1.set_xlabel('Date', fontsize=11, fontweight='bold', color='black')
ax1.set_title('',
              fontsize=12, fontweight='bold', color='black', pad=8)
ax1.text(BREAK + pd.DateOffset(months=2),
         ax1.get_ylim()[0] * 0.80, '1975 Q1',
         fontsize=7.5, color=AMBER)
ax1.legend(fontsize=8.5, frameon=False, loc='lower left')


plt.tight_layout()
plt.savefig('q1_plot1_gaps_compared.pdf', format='pdf', bbox_inches='tight')
print("Plot 1 saved: q1_plot1_gaps_compared.pdf")

# =============================================================================
# 9. FIGURE 2 — Sample sensitivity: MN and Log-linear only
# =============================================================================
fig2, axes2 = plt.subplots(2, 1, figsize=(12, 9), dpi=130)
fig2.patch.set_facecolor('white')

panel_titles = ['McCallum-Nelson Gap', 'Log-linear Trend Gap']
gap_keys     = ['gap1', 'gap2']

for ax, title, key in zip(axes2, panel_titles, gap_keys):
    style_ax(ax)
    shade_recessions(ax, df_all.index)
    ax.set_xlabel('Date', fontsize=11, fontweight='bold', color='black')
    ax.set_ylabel('Output gap (%)', fontsize=11, fontweight='bold', color='black')
    ax.set_title(title, fontsize=12, fontweight='bold', color='black', pad=7)
    ax.text(BREAK + pd.DateOffset(months=2), -13,
            '1975 Q1', fontsize=7.5, color=AMBER)

    for label, d in sensitivity.items():
        series = d[key] * 100
        ls, lw, col, zo = LINE_STYLES[label]
        ax.plot(series.index, series,
                color=col, lw=lw, ls=ls,
                label=label, zorder=zo)

    ax.legend(fontsize=8, frameon=False, loc='lower left')
    ax.set_ylim(-15, 12)

fig2.suptitle('',
              fontsize=13, fontweight='bold', color='black', y=1.01)

plt.tight_layout(h_pad=3.2)
plt.savefig('q1_plot2_sensitivity.pdf', format='pdf', bbox_inches='tight')
print("Plot 2 saved: q1_plot2_sensitivity.pdf")

# =============================================================================
# 10. PRINT KEY STATISTICS FOR WRITEUP
# =============================================================================
print("\n=== KEY STATISTICS FOR WRITEUP ===")
for gap, label in [('gap_MN',    'MN Labour gap'),
                   ('gap_trend', 'Log-linear trend'),
                   ('gap_hp',    'HP filter'),
                   ('gap_bp',    'Band-pass')]:
    s_full = full[gap].dropna() * 100
    s_var  = full.loc['1966-01-01':'2000-10-01', gap].dropna() * 100
    print(f"\n{label}:")
    print(f"  Full sample: min={s_full.min():.2f}%  max={s_full.max():.2f}%  "
          f"std={s_full.std():.2f}%")
    print(f"  VAR window (1966–2000): mean={s_var.mean():.3f}%  "
          f"std={s_var.std():.3f}%")

print("\n=== SENSITIVITY SUMMARY ===")
for key, gaplabel in [('gap1', 'MN gap'), ('gap2', 'Log-linear gap')]:
    print(f"\n{gaplabel} — range of estimates at 1990 Q1 across windows:")
    vals = []
    for label, d in sensitivity.items():
        s = d[key] * 100
        if pd.Timestamp('1990-01-01') in s.index:
            vals.append((label, s.loc['1990-01-01']))
    for label, v in vals:
        print(f"  {label}: {v:.2f}%")
    if vals:
        spread = max(v for _, v in vals) - min(v for _, v in vals)
        print(f"  Spread at 1990 Q1: {spread:.2f}pp")

        # Print MN gap at key recession troughs
mn = full['gap_MN'] * 100
print("MN gap at key dates:")
for date in ['1975-01-01', '1982-10-01', '1991-01-01',
             '2009-04-01', '2020-04-01']:
    if date in mn.index:
        print(f"  {date}: {mn.loc[date]:.2f}%")
print(f"  Min date: {mn.idxmin().date()} val={mn.min():.2f}%")
print(f"  Max date: {mn.idxmax().date()} val={mn.max():.2f}%")

Log-linear trend estimated on full sample: 1950-01-01 to 2024-10-01, N=300
  R²=0.9957  pre-break slope=0.009580  post-break slope change=-0.002864

MN gap: n̄ = -7.3234 (mean of n_t over 1950–2000)

Full sample: 1950-01-01 to 2024-10-01
Band-pass valid: 1953-01-01 to 2021-10-01 (loses 12 qtrs each end)
  Full sample (from 1950): 1950-01-01 to 2024-10-01, N=300
  From 1980: 1980-01-01 to 2024-10-01, N=180
  From 1966: 1966-01-01 to 2024-10-01, N=236

=== Summary statistics 1966 Q1–2000 Q4 ===
         gap_MN  gap_trend    gap_hp    gap_bp
count  140.0000   140.0000  140.0000  140.0000
mean    -0.0160     0.0024    0.0010    0.0013
std      0.0316     0.0400    0.0163    0.0154
min     -0.0817    -0.0927   -0.0480   -0.0439
25%     -0.0402    -0.0165   -0.0085   -0.0065
50%     -0.0205     0.0035    0.0014    0.0008
75%      0.0037     0.0278    0.0115    0.0115
max      0.0584     0.0971    0.0372    0.0342

Correlation matrix:
           gap_MN  gap_trend  gap_hp  gap_bp
gap_MN      1

In [216]:
"""
Part 2 — Structural VAR(4) Estimation and Impulse Response Analysis
Replicates the Estrella & Fuhrer (2002) Figure 2 framework
Cholesky ordering: x (defense) → π (inflation) → y (output gap) → r (FFR)
Unit shock normalisation: all IRFs divided by FFR impact at h=0 → 1pp shock
Sample: 1966 Q1 – 2000 Q4
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from statsmodels.tsa.vector_ar.var_model import VAR
import warnings
import openpyxl
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD DATA
# =============================================================================
wb = openpyxl.load_workbook('EC924_2025_Asiifnment_DataDistribution.xlsx')

ws = wb['Quarterly']
rows = list(ws.iter_rows(values_only=True))
df = pd.DataFrame(rows[1:], columns=rows[0])
df = df.dropna(subset=['observation_date'])
df['date'] = pd.to_datetime(df['observation_date'])
df = df.set_index('date')

ws_m   = wb['Monthly']
rows_m = list(ws_m.iter_rows(values_only=True))
dfm    = pd.DataFrame(rows_m[1:], columns=rows_m[0])
dfm    = dfm.dropna(subset=['observation_date'])
dfm['date'] = pd.to_datetime(dfm['observation_date'])
dfm    = dfm.set_index('date')
clf_q  = dfm['CLF16OV'].resample('QS').mean()
df['clf'] = clf_q

# =============================================================================
# 2. CONSTRUCT VARIABLES
# =============================================================================
df['log_fdefx'] = np.log(df['FDEFX'])
df['infl']      = np.log(df['USAGDPDEFQISMEI']).diff()
df['ffr']       = df['DFF']
df['n_t']       = np.log(df['HOANBS']) - np.log(df['clf'])

# =============================================================================
# 3. OUTPUT GAP — MN SAMPLE MEAN METHOD
# ỹ_t = 0.7 × (n_t − n̄)
# n̄ = sample mean over VAR window 1966 Q1–2000 Q4
# =============================================================================
samp  = df[['n_t']].dropna().loc['1966-01-01':'2000-10-01'].copy()
n_bar = samp['n_t'].mean()

df['gap_MN'] = np.nan
full_nt = df[['n_t']].dropna()
df.loc[full_nt.index, 'gap_MN'] = 0.7 * (full_nt['n_t'] - n_bar)

print("Output gap constructed (sample mean method).")
print(f"  n̄ (sample mean of n_t, 1966–2000): {n_bar:.4f}")
print(f"  gap_MN range: {df['gap_MN'].min():.4f} to {df['gap_MN'].max():.4f}")
print(f"  Non-null observations: {df['gap_MN'].notna().sum()}")

# =============================================================================
# 4. VAR ESTIMATION
# =============================================================================
var_vars = ['log_fdefx', 'infl', 'gap_MN', 'ffr']
var_data = df[var_vars].dropna().loc['1966-01-01':'2000-10-01'].copy()
var_data.columns = ['log_fdefx', 'infl', 'gap_MN', 'ffr']
print(f"\nVAR sample: {var_data.index[0].date()} to {var_data.index[-1].date()}")
print(f"Observations: {len(var_data)}")
print(f"Any NaNs: {var_data.isnull().any().any()}")

var_result = VAR(var_data).fit(4)

# =============================================================================
# 5. IMPULSE RESPONSE FUNCTIONS
# =============================================================================
H       = 40
irf_obj = var_result.irf(H)
irfs    = irf_obj.orth_irfs    # (H+1, 4, 4)

# =============================================================================
# 5b. BOOTSTRAP CI
# =============================================================================
n_reps = 500
rng    = np.random.default_rng(42)

lags      = var_result.k_ar
resids    = var_result.resid.values
params    = var_result.params.values
T_resid   = resids.shape[0]
k         = resids.shape[1]
data_orig = var_data.values
boot_index = var_data.index[lags:]
boot_irfs  = np.zeros((n_reps, H + 1, k, k))
n_failed   = 0

print(f"Running {n_reps} bootstrap replications...")

for b in range(n_reps):
    idx      = rng.integers(0, T_resid, size=T_resid)
    resids_b = resids[idx]
    data_b   = list(data_orig[:lags])
    for t in range(T_resid):
        row     = np.concatenate([[1.0]] +
                                 [data_b[-(lag+1)] for lag in range(lags)])
        new_obs = row @ params + resids_b[t]
        data_b.append(new_obs)
    data_b = np.array(data_b)[lags:]
    try:
        df_b  = pd.DataFrame(data_b,
                             columns=var_data.columns,
                             index=boot_index)
        res_b = VAR(df_b).fit(lags)
        irf_b = res_b.irf(H)
        boot_irfs[b] = irf_b.orth_irfs
    except Exception:
        boot_irfs[b] = np.nan
        n_failed += 1

print(f"Failed replications: {n_failed} / {n_reps}")
valid     = ~np.isnan(boot_irfs[:, 0, 0, 0])
boot_irfs = boot_irfs[valid]
print(f"Valid replications: {valid.sum()} / {n_reps}")

lo90 = np.percentile(boot_irfs,  5, axis=0)
hi90 = np.percentile(boot_irfs, 95, axis=0)

# =============================================================================
# 5c. UNIT SHOCK NORMALISATION
# Divide all IRFs by FFR impact at h=0 → monetary shock = exactly 1pp FFR
# Consistent with E&F and standard comparison convention
# =============================================================================
var_r_impact_raw = irfs[0, 3, 3]
norm_factor      = 1.0 / var_r_impact_raw

irfs_norm = irfs * norm_factor
lo90_norm = lo90 * norm_factor
hi90_norm = hi90 * norm_factor

print(f"\nUnit shock normalisation:")
print(f"  Raw FFR impact at h=0: {var_r_impact_raw:.4f}pp")
print(f"  Norm factor: {norm_factor:.4f}")
print(f"  FFR at h=0 after norm: {irfs_norm[0,3,3]:.4f}pp  ← should be 1.000")

# =============================================================================
# 6. SCALING AND LABELS
# =============================================================================
scales     = [1,   400,   100,   1]
var_labels = ['Log real defense exp.',
              'Inflation (ann. pp)',
              'Output gap (pp)',
              'Fed funds rate (pp)']
shock_labs = ['Defense Shock',
              'Inflation Shock',
              'Output Gap Shock',
              'Monetary Shock']

quarters = np.arange(H + 1)
shock_j  = 3    # monetary shock index

# =============================================================================
# 7. PRINT KEY IRF VALUES FOR WRITEUP (unit shock normalised)
# =============================================================================
print("\n=== KEY IRF VALUES: Monetary shock (unit shock, 1pp FFR) ===")
for resp_i, (label, sc) in enumerate(zip(var_labels, scales)):
    line = irfs_norm[:, resp_i, shock_j] * sc
    peak_h = int(np.argmin(line)) if resp_i == 2 else int(np.argmax(np.abs(line)))
    print(f"\n{label}:")
    print(f"  h=0: {line[0]:.4f}  h=1: {line[1]:.4f}  h=2: {line[2]:.4f}  "
          f"h=4: {line[4]:.4f}  h=8: {line[8]:.4f}  h=12: {line[12]:.4f}  "
          f"peak h={peak_h}: {line[peak_h]:.4f}")

# =============================================================================
# 8. PLOT: 3×3 IRF grid matching E&F Figure 2 layout (unit shock normalised)
# =============================================================================
var_indices   = [1, 3, 2]
shock_indices = [1, 3, 2]

row_labels_3x3 = [var_labels[i] for i in var_indices]
col_labels_3x3 = [shock_labs[i] for i in shock_indices]

blue_line  = '#185FA5'
gray_zero  = '#888780'
fill_color = '#E0E0E0'

fig, axes = plt.subplots(3, 3, figsize=(11, 9), dpi=130)
fig.patch.set_facecolor('white')

for col, sj in enumerate(shock_indices):
    for row, resp_i in enumerate(var_indices):
        ax   = axes[row, col]
        sc   = scales[resp_i]
        line = irfs_norm[:, resp_i, sj] * sc    # normalised
        lo   = lo90_norm[:, resp_i, sj] * sc    # normalised
        hi   = hi90_norm[:, resp_i, sj] * sc    # normalised

        ax.fill_between(quarters, lo, hi,
                        color=fill_color, alpha=0.8, edgecolor='none')
        ax.plot(quarters, line, color=blue_line, lw=2.0)
        ax.axhline(0, color=gray_zero, lw=0.8, ls='--', alpha=0.5)

        ax.set_facecolor('white')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#D3D1C7')
        ax.spines['bottom'].set_color('#D3D1C7')
        ax.tick_params(labelsize=9, colors='black')
        ax.set_xlim(0, H)
        ax.set_xticks([0, 10, 20, 30, 40])
        ax.margins(x=0)

        if row == 0:
            ax.set_title(col_labels_3x3[col], fontsize=12,
                         fontweight='bold', color='black', pad=8)
        if col == 0:
            ax.set_ylabel(row_labels_3x3[row], fontsize=11,
                          fontweight='bold', color='black')
        if row == 2:
            ax.set_xlabel('Quarters', fontsize=11,
                          fontweight='bold', color='black')

plt.tight_layout()
plt.savefig('q2_irf_3x3.pdf', format='pdf', bbox_inches='tight')
print("Figure saved: q2_irf_3x3.pdf")

# =============================================================================
# 9. PLOT: Monetary shock only — 1×3 panel (unit shock normalised)
# =============================================================================
fig2, axes2 = plt.subplots(1, 3, figsize=(12, 3.8), dpi=130)
fig2.patch.set_facecolor('white')

monetary_shock = 3
response_order = [1, 3, 2]
row_labels_mon = [var_labels[i] for i in response_order]

for col, resp_i in enumerate(response_order):
    ax   = axes2[col]
    sc   = scales[resp_i]
    line = irfs_norm[:, resp_i, monetary_shock] * sc    # normalised
    lo   = lo90_norm[:, resp_i, monetary_shock] * sc    # normalised
    hi   = hi90_norm[:, resp_i, monetary_shock] * sc    # normalised

    ax.fill_between(quarters, lo, hi,
                    color=fill_color, alpha=0.8, edgecolor='none')
    ax.plot(quarters, line, color=blue_line, lw=2.2)
    ax.axhline(0, color=gray_zero, lw=0.8, ls='--', alpha=0.5)

    ax.set_facecolor('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#D3D1C7')
    ax.spines['bottom'].set_color('#D3D1C7')
    ax.tick_params(labelsize=9, colors='black')
    ax.set_xlim(0, H)
    ax.set_xticks([0, 10, 20, 30, 40])
    ax.margins(x=0)

    ax.set_xlabel('Quarters after shock', fontsize=11,
                  fontweight='bold', color='black')
    ax.set_title(row_labels_mon[col], fontsize=12,
                 fontweight='bold', color='black', pad=8)

plt.tight_layout()
plt.savefig('q2_monetary_shock_irf.pdf', format='pdf', bbox_inches='tight')
print("Figure saved: q2_monetary_shock_irf.pdf")

Output gap constructed (sample mean method).
  n̄ (sample mean of n_t, 1966–2000): -7.3463
  gap_MN range: -0.1074 to 0.0921
  Non-null observations: 300

VAR sample: 1966-01-01 to 2000-10-01
Observations: 140
Any NaNs: False
Running 500 bootstrap replications...
Failed replications: 0 / 500
Valid replications: 500 / 500

Unit shock normalisation:
  Raw FFR impact at h=0: 0.8577pp
  Norm factor: 1.1658
  FFR at h=0 after norm: 1.0000pp  ← should be 1.000

=== KEY IRF VALUES: Monetary shock (unit shock, 1pp FFR) ===

Log real defense exp.:
  h=0: 0.0000  h=1: 0.0038  h=2: 0.0070  h=4: 0.0078  h=8: 0.0143  h=12: 0.0182  peak h=27: 0.0224

Inflation (ann. pp):
  h=0: 0.0000  h=1: 0.3666  h=2: 0.3086  h=4: 0.1154  h=8: 0.0345  h=12: -0.0195  peak h=1: 0.3666

Output gap (pp):
  h=0: 0.0000  h=1: 0.0869  h=2: -0.0154  h=4: -0.0894  h=8: -0.2669  h=12: -0.2748  peak h=10: -0.2913

Fed funds rate (pp):
  h=0: 1.0000  h=1: 0.9251  h=2: 0.4782  h=4: 0.6732  h=8: 0.3053  h=12: 0.1308  peak h=0: 

In [212]:
"""
Part 3 — Rational Expectations (DSGE) Model Solution
Closed-form IRFs via the Blanchard-Kahn method, benchmarked against the VAR.
Gap: 0.7 × (n_t − n̄), n̄ = sample mean over VAR window 1966 Q1–2000 Q4
Unit shock normalisation: all VAR IRFs divided by FFR impact at h=0 → 1pp shock
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from statsmodels.tsa.vector_ar.var_model import VAR
import warnings
import openpyxl
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD DATA
# =============================================================================
wb = openpyxl.load_workbook('EC924_2025_Asiifnment_DataDistribution.xlsx')

ws = wb['Quarterly']
rows = list(ws.iter_rows(values_only=True))
df = pd.DataFrame(rows[1:], columns=rows[0])
df = df.dropna(subset=['observation_date'])
df['date'] = pd.to_datetime(df['observation_date'])
df = df.set_index('date')

ws_m   = wb['Monthly']
rows_m = list(ws_m.iter_rows(values_only=True))
dfm    = pd.DataFrame(rows_m[1:], columns=rows_m[0])
dfm    = dfm.dropna(subset=['observation_date'])
dfm['date'] = pd.to_datetime(dfm['observation_date'])
dfm    = dfm.set_index('date')
clf_q  = dfm['CLF16OV'].resample('QS').mean()
df['clf'] = clf_q

df['lngdp']     = np.log(df['GDPC1'])
df['log_fdefx'] = np.log(df['FDEFX'])
df['infl']      = np.log(df['USAGDPDEFQISMEI']).diff()
df['ffr']       = df['DFF']

# =============================================================================
# 2. OUTPUT GAP
# =============================================================================
df['n_t'] = np.log(df['HOANBS']) - np.log(df['clf'])
n_bar = df['n_t'].dropna().loc['1966-01-01':'2000-10-01'].mean()
df['gap_main'] = 0.7 * (df['n_t'] - n_bar)
df['gap_sub']  = 0.7 * (df['n_t'] - n_bar)

print(f"n̄ (sample mean, 1966–2000): {n_bar:.4f}")
print(f"gap_main range: {df['gap_main'].dropna().min():.4f} "
      f"to {df['gap_main'].dropna().max():.4f}")

# =============================================================================
# 3. BOOTSTRAP CI FUNCTION
# =============================================================================
def manual_bootstrap_ci(var_result, H=40, n_reps=500, seed=42,
                         pct_lo=5, pct_hi=95):
    rng        = np.random.default_rng(seed)
    lags       = var_result.k_ar
    resids     = var_result.resid.values
    params     = var_result.params.values
    T_resid    = resids.shape[0]
    k          = resids.shape[1]
    data_orig  = var_result.model.endog
    boot_index = var_result.model.data.dates[lags:]
    boot_irfs  = np.zeros((n_reps, H + 1, k, k))
    for b in range(n_reps):
        idx      = rng.integers(0, T_resid, size=T_resid)
        resids_b = resids[idx]
        data_b   = list(data_orig[:lags])
        for t in range(T_resid):
            row = np.concatenate([[1.0]] +
                                 [data_b[-(lag+1)] for lag in range(lags)])
            data_b.append(row @ params + resids_b[t])
        data_b = np.array(data_b)[lags:]
        try:
            df_b  = pd.DataFrame(data_b,
                                 columns=var_result.model.endog_names,
                                 index=boot_index)
            res_b = VAR(df_b).fit(lags)
            boot_irfs[b] = res_b.irf(H).orth_irfs
        except Exception:
            boot_irfs[b] = np.nan
    valid     = ~np.isnan(boot_irfs[:, 0, 0, 0])
    boot_irfs = boot_irfs[valid]
    print(f"  Valid replications: {valid.sum()} / {n_reps}")
    return (np.percentile(boot_irfs, pct_lo, axis=0),
            np.percentile(boot_irfs, pct_hi, axis=0))

# =============================================================================
# 4. VAR ESTIMATION
# =============================================================================
H        = 40
quarters = np.arange(H + 1)

vm = df[['log_fdefx', 'infl', 'gap_main', 'ffr']].dropna().loc[
     '1966-01-01':'2000-10-01'].copy()
vm.columns = ['log_fdefx', 'infl', 'gap_MN', 'ffr']
print(f"Main VAR: {vm.index[0].date()} to {vm.index[-1].date()}, N={len(vm)}")

res_m  = VAR(vm).fit(4)
irf_m  = res_m.irf(H)
irfs_m = irf_m.orth_irfs
print("Computing main VAR bootstrap CI...")
lo_m, hi_m = manual_bootstrap_ci(res_m, H=H)

vs = df[['log_fdefx', 'infl', 'gap_sub', 'ffr']].dropna().loc[
     '1985-01-01':'2006-10-01'].copy()
vs.columns = ['log_fdefx', 'infl', 'gap_MN', 'ffr']
print(f"\nSubsample VAR: {vs.index[0].date()} to {vs.index[-1].date()}, N={len(vs)}")

res_s  = VAR(vs).fit(4)
irf_s  = res_s.irf(H)
irfs_s = irf_s.orth_irfs
print("Computing subsample VAR bootstrap CI...")
lo_s, hi_s = manual_bootstrap_ci(res_s, H=H)

# =============================================================================
# 4b. UNIT SHOCK NORMALISATION
# Both VARs normalised by the MAIN VAR's FFR impact at h=0
# This puts both on the same 1pp scale for direct comparison
# =============================================================================
var_r_impact_raw = irfs_m[0, 3, 3]
norm_m           = 1.0 / var_r_impact_raw

# Main VAR — normalise by own FFR impact
irfs_m_norm = irfs_m * norm_m
lo_m_norm   = lo_m   * norm_m
hi_m_norm   = hi_m   * norm_m

# Subsample VAR — normalise by SAME factor as main VAR (not its own)
# This keeps both VARs on the same 1pp scale for direct comparison
irfs_s_norm = irfs_s * norm_m    # ← same norm_m, not norm_s
lo_s_norm   = lo_s   * norm_m    # ← same norm_m
hi_s_norm   = hi_s   * norm_m    # ← same norm_m

print(f"\nUnit shock normalisation:")
print(f"  Main VAR FFR impact (raw): {var_r_impact_raw:.4f}pp  "
      f"norm factor: {norm_m:.4f}")
print(f"  Subsample FFR impact (raw): {irfs_s[0,3,3]:.4f}pp  "
      f"(normalised by main factor for comparability)")
print(f"  Main VAR FFR at h=0 after norm: "
      f"{irfs_m_norm[0,3,3]:.4f}pp  ← should be 1.000")

# MN model normalised to 1pp FFR shock
var_r_impact_mn = 1.0

# =============================================================================
# 5. MN MODEL — BLANCHARD-KAHN SOLUTION
# =============================================================================
beta     = 0.99
kappa    = 0.075
sigma    = 0.164
rho      = 0.69
theta_pi = 0.31 * 1.91
theta_y  = 0.31 * 0.07

def residual(phi_r):
    d = (1 - phi_r) * (1 - beta * phi_r)
    if abs(d) < 1e-10: return 1e6
    a  = kappa * sigma * phi_r / d
    if abs(a - 1) < 1e-10: return 1e6
    P0 = a / (a - 1)
    P1 = phi_r * sigma * (P0 - 1) / (1 - phi_r)
    return (rho + theta_y * P1) / (1 - theta_pi * P0) - phi_r

phi_r = brentq(residual, 0.60, 0.70)
a     = kappa * sigma * phi_r / ((1 - phi_r) * (1 - beta * phi_r))
P0    = a / (a - 1)
P1    = phi_r * sigma * (P0 - 1) / (1 - phi_r)

K       = P1 + sigma * (P0 - 1)
c       = 1 - theta_pi * P0
denom   = c - theta_y * K
e_y     = np.array([0., 1., 0.])
e_r     = np.array([0., 0., 1.])
e_pi    = np.array([1., 0., 0.])
phi_eps = (theta_y * e_y + e_r) / denom
Q_y     = phi_eps * K + e_y
Q_pi    = beta * P0 * phi_eps + kappa * Q_y + e_pi

print(f"\nMN model solution: phi_r = {phi_r:.4f}")
print(f"  P0 = {P0:.4f}, P1 = {P1:.4f}")

# =============================================================================
# 6. COMPUTE MN MODEL IRFs
# Normalised to 1pp FFR unit shock (var_r_impact_mn = 1.0)
# mn_y in model pp units — no ×4
# =============================================================================
j        = 2
scale_mn = var_r_impact_mn / (phi_eps[j] * 4)    # normalise to 1pp FFR

mn_r  = np.zeros(H + 1)
mn_pi = np.zeros(H + 1)
mn_y  = np.zeros(H + 1)

mn_r[0]  = phi_eps[j] * scale_mn * 4
mn_pi[0] = Q_pi[j]    * scale_mn * 4
mn_y[0]  = Q_y[j]     * scale_mn        # model pp — no ×4

for h in range(1, H + 1):
    r_prev   = phi_eps[j] * scale_mn * phi_r ** (h - 1)
    mn_r[h]  = r_prev * phi_r * 4
    mn_pi[h] = P0 * r_prev * 4
    mn_y[h]  = P1 * r_prev              # model pp — no ×4

print(f"\nMN IRF at impact (unit shock): r={mn_r[0]:.4f}pp  "
      f"y={mn_y[0]:.4f}pp  pi={mn_pi[0]:.4f}ann.pp")
print(f"VAR IRF at impact (normalised): r={irfs_m_norm[0,3,3]:.4f}pp  "
      f"y={irfs_m_norm[0,2,3]*100:.4f}pp  "
      f"pi={irfs_m_norm[0,1,3]*400:.4f}ann.pp")

# =============================================================================
# 7. SCALING AND LABELS
# =============================================================================
scales     = [1,   400,   100,   1]
var_labels = ['Log real defense exp.',
              'Inflation (ann. pp)',
              'Output gap (pp)',
              'Fed funds rate (pp)']

var_indices    = [1, 3, 2]
row_labels_3x3 = [var_labels[i] for i in var_indices]

mn_by_varidx = {
    1: mn_pi,
    3: mn_r,
    2: mn_y,    # model pp matching normalised VAR gap units
}

monetary_shock = 3

# =============================================================================
# 8. FIGURE A — VAR vs MN model (unit shock normalised)
# =============================================================================
blue_line = '#185FA5'
red_mn    = '#C04828'
gray_zero = '#888780'

response_order = [1, 3, 2]
row_labels_mon = [var_labels[i] for i in response_order]

fig_a, axes_a = plt.subplots(1, 3, figsize=(14, 4.5), dpi=130)
fig_a.patch.set_facecolor('white')

for col, resp_i in enumerate(response_order):
    ax   = axes_a[col]
    sc   = scales[resp_i]
    line = irfs_m_norm[:, resp_i, monetary_shock] * sc    # normalised
    lo   = lo_m_norm[:,  resp_i, monetary_shock] * sc     # normalised
    hi   = hi_m_norm[:,  resp_i, monetary_shock] * sc     # normalised
    mn   = mn_by_varidx[resp_i]

    ax.fill_between(quarters, lo, hi, color='#B0B0B0', alpha=0.18,
                    edgecolor='none', label='VAR 90% CI')
    ax.plot(quarters, line, color=blue_line, lw=2.0, label='VAR (1966–2000)')
    ax.plot(quarters, mn,   color=red_mn,   lw=2.0, ls='--', label='MN Model')
    ax.axhline(0, color=gray_zero, lw=0.8, ls='--', alpha=0.5)

    ax.set_facecolor('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#D3D1C7')
    ax.spines['bottom'].set_color('#D3D1C7')
    ax.tick_params(labelsize=10, colors='black')
    ax.set_xlim(0, H)
    ax.set_xticks([0, 10, 20, 30, 40])
    ax.margins(x=0)

    if resp_i == 1:
        ax.set_ylim(-0.25, 0.65)
    elif resp_i == 3:
        ax.set_ylim(-0.30, 1.30)
    elif resp_i == 2:
        ax.set_ylim(-0.50, 0.15)

    ax.set_xlabel('Quarters', fontsize=11, fontweight='bold', color='black')
    ax.set_title(row_labels_mon[col], fontsize=12, fontweight='bold',
                 color='black', pad=8)

axes_a[2].legend(fontsize=8.5, frameon=False, loc='lower right',
                 borderaxespad=0.5)
plt.tight_layout(w_pad=0.5)
plt.savefig('q3_var_vs_mn.pdf', format='pdf', bbox_inches='tight')
print("Figure A saved: q3_var_vs_mn.pdf")

# =============================================================================
# 9. FIGURE B — Subsample comparison (unit shock normalised)
# =============================================================================
purp_line = '#2E7D32'

fig_b, axes_b = plt.subplots(1, 3, figsize=(14, 4.5), dpi=130)
fig_b.patch.set_facecolor('white')

for col, resp_i in enumerate(response_order):
    ax     = axes_b[col]
    sc     = scales[resp_i]
    line_m = irfs_m_norm[:, resp_i, monetary_shock] * sc    # normalised main
    line_s = irfs_s_norm[:, resp_i, monetary_shock] * sc    # normalised subsample
    mn     = mn_by_varidx[resp_i]

    ax.plot(quarters, line_m, color=blue_line, lw=2.0, label='VAR (1966–2000)')
    ax.plot(quarters, line_s, color=purp_line, lw=2.0, ls='--',
            label='VAR (1985–2006)')
    ax.plot(quarters, mn,     color=red_mn,   lw=1.8, ls=':', label='MN Model')
    ax.axhline(0, color=gray_zero, lw=0.8, ls='--', alpha=0.5)

    ax.set_facecolor('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#D3D1C7')
    ax.spines['bottom'].set_color('#D3D1C7')
    ax.tick_params(labelsize=10, colors='black')
    ax.set_xlim(0, H)
    ax.set_xticks([0, 10, 20, 30, 40])
    ax.margins(x=0)

    if resp_i == 1:
        ax.set_ylim(-0.25, 0.60)
    elif resp_i == 3:
        ax.set_ylim(-0.30, 1.30)
    elif resp_i == 2:
        ax.set_ylim(-0.50, 0.35)

    ax.set_xlabel('Quarters', fontsize=11, fontweight='bold', color='black')
    ax.set_title(row_labels_mon[col], fontsize=12, fontweight='bold',
                 color='black', pad=8)

axes_b[2].legend(fontsize=8.5, frameon=False, loc='lower right',
                 borderaxespad=0.5)
plt.tight_layout(w_pad=0.5)
plt.savefig('q3_subsample_comparison.pdf', format='pdf', bbox_inches='tight')
print("Figure B saved: q3_subsample_comparison.pdf")

# =============================================================================
# 10. PRINT KEY NUMBERS FOR WRITEUP
# =============================================================================
print("\n=== KEY NUMBERS ===")
print(f"MN phi_r = {phi_r:.4f} "
      f"(half-life = {np.log(0.5)/np.log(phi_r):.1f} quarters)")

print(f"\nVAR main — monetary shock → output gap (unit shock normalised):")
print(f"  h=0:{irfs_m_norm[0,2,3]*100:.3f}pp  "
      f"h=2:{irfs_m_norm[2,2,3]*100:.3f}pp  "
      f"h=8:{irfs_m_norm[8,2,3]*100:.3f}pp  "
      f"trough h={np.argmin(irfs_m_norm[:,2,3])}  "
      f"val={irfs_m_norm[:,2,3].min()*100:.3f}pp")

print(f"\nMN model — monetary shock → output gap (unit shock):")
print(f"  h=0:{mn_y[0]:.3f}pp  h=2:{mn_y[2]:.3f}pp  "
      f"h=8:{mn_y[8]:.3f}pp")

print(f"\nVAR subsample — monetary shock → output gap (unit shock normalised):")
print(f"  h=0:{irfs_s_norm[0,2,3]*100:.3f}pp  "
      f"h=2:{irfs_s_norm[2,2,3]*100:.3f}pp  "
      f"h=8:{irfs_s_norm[8,2,3]*100:.3f}pp  "
      f"trough h={np.argmin(irfs_s_norm[:,2,3])}  "
      f"val={irfs_s_norm[:,2,3].min()*100:.3f}pp")

print(f"\nVAR main — monetary shock → inflation (unit shock):")
print(f"  h=0:{irfs_m_norm[0,1,3]*400:.3f}pp  "
      f"h=1:{irfs_m_norm[1,1,3]*400:.3f}pp")

print(f"\nVAR main — monetary shock → FFR (unit shock):")
print(f"  h=0:{irfs_m_norm[0,3,3]:.3f}pp  "
      f"h=1:{irfs_m_norm[1,3,3]:.3f}pp")

print(f"\nMN model — monetary shock → FFR:")
print(f"  h=0:{mn_r[0]:.3f}pp  ← should equal 1.000")

n̄ (sample mean, 1966–2000): -7.3463
gap_main range: -0.1074 to 0.0921
Main VAR: 1966-01-01 to 2000-10-01, N=140
Computing main VAR bootstrap CI...
  Valid replications: 500 / 500

Subsample VAR: 1985-01-01 to 2006-10-01, N=88
Computing subsample VAR bootstrap CI...
  Valid replications: 500 / 500

Unit shock normalisation:
  Main VAR FFR impact (raw): 0.8577pp  norm factor: 1.1658
  Subsample FFR impact (raw): 0.3127pp  (normalised by main factor for comparability)
  Main VAR FFR at h=0 after norm: 1.0000pp  ← should be 1.000

MN model solution: phi_r = 0.6551
  P0 = -0.0712, P1 = -0.3337

MN IRF at impact (unit shock): r=1.0000pp  y=-0.1274pp  pi=-0.1087ann.pp
VAR IRF at impact (normalised): r=1.0000pp  y=0.0000pp  pi=0.0000ann.pp
Figure A saved: q3_var_vs_mn.pdf
Figure B saved: q3_subsample_comparison.pdf

=== KEY NUMBERS ===
MN phi_r = 0.6551 (half-life = 1.6 quarters)

VAR main — monetary shock → output gap (unit shock normalised):
  h=0:0.000pp  h=2:-0.015pp  h=8:-0.267pp  trough

In [211]:
"""
Part 4 — Extended DSGE Model: Habit Formation and Hybrid Phillips Curve
Two separate analyses:
  Figure A — Output gap focus: Base MN, Habit IS, Full model (γ=0.90, α=0.55)
  Figure B — Inflation focus:  Base MN, Hybrid PC (γ=0.50) — two lines only
Gap: 0.7 × (n_t − n̄), n̄ = sample mean over VAR window 1966 Q1–2000 Q4
Unit shock normalisation: VAR IRFs divided by FFR impact at h=0 → 1pp shock
MN model: normalised to same 1pp FFR unit shock
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from statsmodels.tsa.vector_ar.var_model import VAR
import warnings
import openpyxl
warnings.filterwarnings('ignore')

# =============================================================================
# 1. LOAD DATA
# =============================================================================
wb = openpyxl.load_workbook('EC924_2025_Asiifnment_DataDistribution.xlsx')
ws = wb['Quarterly']
rows = list(ws.iter_rows(values_only=True))
df = pd.DataFrame(rows[1:], columns=rows[0])
df = df.dropna(subset=['observation_date'])
df['date'] = pd.to_datetime(df['observation_date'])
df = df.set_index('date')

df['lngdp']     = np.log(df['GDPC1'])
df['log_fdefx'] = np.log(df['FDEFX'])
df['infl']      = np.log(df['USAGDPDEFQISMEI']).diff()
df['ffr']       = df['DFF']

# =============================================================================
# 2. MN OUTPUT GAP — SAMPLE MEAN METHOD
# ỹ_t = 0.7 × (n_t − n̄)
# n̄ = sample mean over VAR window 1966 Q1–2000 Q4
# Consistent with Q2 and Q3
# =============================================================================
ws_m   = wb['Monthly']
rows_m = list(ws_m.iter_rows(values_only=True))
dfm    = pd.DataFrame(rows_m[1:], columns=rows_m[0])
dfm    = dfm.dropna(subset=['observation_date'])
dfm['date'] = pd.to_datetime(dfm['observation_date'])
dfm    = dfm.set_index('date')
clf_q  = dfm['CLF16OV'].resample('QS').mean()
df['clf'] = clf_q
df['n_t'] = np.log(df['HOANBS']) - np.log(df['clf'])

n_bar = df['n_t'].dropna().loc['1966-01-01':'2000-10-01'].mean()
df['gap_MN'] = 0.7 * (df['n_t'] - n_bar)

print(f"n̄ (sample mean, 1966–2000): {n_bar:.4f}")
print(f"gap_MN range: {df['gap_MN'].dropna().min():.4f} "
      f"to {df['gap_MN'].dropna().max():.4f}")

# =============================================================================
# 3. BOOTSTRAP CI FUNCTION
# =============================================================================
def manual_bootstrap_ci(var_result, H=40, n_reps=500, seed=42,
                         pct_lo=5, pct_hi=95):
    rng        = np.random.default_rng(seed)
    lags       = var_result.k_ar
    resids     = var_result.resid.values
    params     = var_result.params.values
    T_resid    = resids.shape[0]
    k          = resids.shape[1]
    data_orig  = var_result.model.endog
    boot_index = var_result.model.data.dates[lags:]
    boot_irfs  = np.zeros((n_reps, H + 1, k, k))
    for b in range(n_reps):
        idx      = rng.integers(0, T_resid, size=T_resid)
        resids_b = resids[idx]
        data_b   = list(data_orig[:lags])
        for t in range(T_resid):
            row = np.concatenate([[1.0]] +
                                 [data_b[-(lag+1)] for lag in range(lags)])
            data_b.append(row @ params + resids_b[t])
        data_b = np.array(data_b)[lags:]
        try:
            df_b  = pd.DataFrame(data_b,
                                 columns=var_result.model.endog_names,
                                 index=boot_index)
            res_b = VAR(df_b).fit(lags)
            boot_irfs[b] = res_b.irf(H).orth_irfs
        except Exception:
            boot_irfs[b] = np.nan
    valid     = ~np.isnan(boot_irfs[:, 0, 0, 0])
    boot_irfs = boot_irfs[valid]
    print(f"  Valid replications: {valid.sum()} / {n_reps}")
    return (np.percentile(boot_irfs, pct_lo, axis=0),
            np.percentile(boot_irfs, pct_hi, axis=0))

# =============================================================================
# 4. VAR(4) — 1966 Q1–2000 Q4
# =============================================================================
H        = 40
quarters = np.arange(H + 1)

vm = df[['log_fdefx', 'infl', 'gap_MN', 'ffr']].dropna().loc[
    '1966-01-01':'2000-10-01'].copy()
vm.columns = ['log_fdefx', 'infl', 'gap_MN', 'ffr']
print(f"VAR sample: {vm.index[0].date()} to {vm.index[-1].date()}, N={len(vm)}")

res_m  = VAR(vm).fit(4)
irf_m  = res_m.irf(H)
irfs_m = irf_m.orth_irfs

print("Computing bootstrap CI...")
lo_m, hi_m = manual_bootstrap_ci(res_m, H=H)

# =============================================================================
# 4b. UNIT SHOCK NORMALISATION
# Divide VAR IRFs by FFR impact at h=0 → monetary shock = exactly 1pp FFR
# MN model also normalised to same 1pp unit shock
# =============================================================================
var_r_impact_raw = irfs_m[0, 3, 3]
norm_m           = 1.0 / var_r_impact_raw

irfs_m_norm = irfs_m * norm_m
lo_m_norm   = lo_m   * norm_m
hi_m_norm   = hi_m   * norm_m

# MN model normalised to 1pp FFR unit shock
var_r_impact_mn = 1.0

print(f"\nUnit shock normalisation:")
print(f"  Raw FFR impact: {var_r_impact_raw:.4f}pp  "
      f"norm factor: {norm_m:.4f}")
print(f"  FFR at h=0 after norm: {irfs_m_norm[0,3,3]:.4f}pp")

# =============================================================================
# 5. BLANCHARD-KAHN SOLVER
# =============================================================================
kappa    = 0.075
sigma    = 0.164
theta_pi = 0.31 * 1.91
theta_y  = 0.31 * 0.07


def solve_mn(gamma, alpha, rho):
    e_r  = np.array([1., 0., 0.])
    e_y  = np.array([0., 1., 0.])
    e_pi = np.array([0., 0., 1.])

    def residuals(P_vec):
        P = P_vec.reshape(3, 3)
        res_A = (P[2, :] - gamma * (P[2, :] @ P)
                 - (1 - gamma) * e_pi - kappa * P[1, :])
        res_B = (P[1, :] - alpha * e_y
                 - (1 - alpha) * (P[1, :] @ P)
                 + sigma * P[0, :] - sigma * (P[2, :] @ P))
        res_C = (P[0, :] - rho * e_r
                 - theta_pi * (P[2, :] @ P) - theta_y * P[1, :])
        return np.concatenate([res_A, res_B, res_C])

    P_init = np.zeros(9)
    P_init[0] = 0.655; P_init[3] = -0.334; P_init[6] = -0.071
    P_vec = fsolve(residuals, P_init)
    P     = P_vec.reshape(3, 3)

    ev = np.linalg.eigvals(P)
    if np.max(np.abs(ev)) >= 1.0:
        print(f"  WARNING: unstable — max|λ| = {np.max(np.abs(ev)):.3f}")

    M_block = np.array([
        [1 - theta_pi * P[2, 0],
         -(theta_y + theta_pi * P[2, 1]),
         -theta_pi * P[2, 2]],
        [sigma - (1-alpha)*P[1, 0] - sigma*P[2, 0],
         1 - (1-alpha)*P[1, 1] - sigma*P[2, 1],
         -(1-alpha)*P[1, 2] - sigma*P[2, 2]],
        [-gamma * P[2, 0],
         -kappa - gamma * P[2, 1],
         1 - gamma * P[2, 2]]
    ])
    RHS = np.array([[0., 0., 1.],
                    [0., 1., 0.],
                    [1., 0., 0.]])
    Q = np.linalg.solve(M_block, RHS)
    return P, Q


def compute_irfs(P, Q, var_r_impact, H=40, j=2):
    """
    Compute IRFs to monetary shock normalised to var_r_impact at h=0.
    With var_r_impact=1.0 (unit shock), MN model matches VAR normalisation.
    mn_y in model pp units — no ×4.
    """
    scale = var_r_impact / (Q[0, j] * 4)
    k = Q[:, j] * scale
    mn_r  = np.zeros(H + 1)
    mn_y  = np.zeros(H + 1)
    mn_pi = np.zeros(H + 1)
    for h in range(H + 1):
        mn_r[h]  = k[0] * 4
        mn_y[h]  = k[1]        # model pp — no ×4
        mn_pi[h] = k[2] * 4
        k = P @ k
    return mn_r, mn_y, mn_pi

# =============================================================================
# 6. EXTRACT VAR IRFs — MONETARY SHOCK (from normalised arrays)
# =============================================================================
shock_idx = 3

var_pi = irfs_m_norm[:H+1, 1, shock_idx] * 400    # normalised
var_y  = irfs_m_norm[:H+1, 2, shock_idx] * 100    # normalised
var_r  = irfs_m_norm[:H+1, 3, shock_idx]          # normalised = 1.0 at h=0

print(f"\nVAR (unit shock): y_trough={var_y.min():.3f}pp at h={var_y.argmin()}")
print(f"VAR FFR at h=0: {var_r[0]:.3f}pp  ← should be 1.000")

# =============================================================================
# 7. MODEL CONFIGURATIONS
# =============================================================================
configs_A = [
    ('Base MN',                          1.00,  0.00,  0.69),
    ('Habit IS  (α=0.55)',               1.00,  0.55,  0.69),
    ('Full model  (α=0.55, γ=0.90)',     0.90,  0.55,  0.69),
]

configs_B = [
    ('Base MN',              1.00,  0.00,  0.69),
    ('Hybrid PC  (γ=0.50)',  0.50,  0.00,  0.69),
]

# =============================================================================
# 8. COMPUTE IRFS FOR BOTH CONFIG SETS
# Pass var_r_impact_mn=1.0 so MN is on same 1pp unit shock scale as VAR
# =============================================================================
def compute_config_irfs(configs, var_r_impact, H):
    irfs_out = {}
    print(f"\n{'Model':35s}  {'max|λ|':>7}  {'y_trough':>10}  {'h':>4}")
    for name, gamma, alpha, rho in configs:
        P, Q = solve_mn(gamma, alpha, rho)
        mn_r, mn_y, mn_pi = compute_irfs(P, Q, var_r_impact, H=H)
        irfs_out[name] = dict(r=mn_r, y=mn_y, pi=mn_pi)
        ev = np.max(np.abs(np.linalg.eigvals(P)))
        print(f"  {name:33s}  {ev:>7.3f}  "
              f"{mn_y.min():>10.3f}pp  h={mn_y.argmin()}")
    return irfs_out

print("=== Figure A models ===")
irfs_A = compute_config_irfs(configs_A, var_r_impact_mn, H)

print("\n=== Figure B models ===")
irfs_B = compute_config_irfs(configs_B, var_r_impact_mn, H)

# =============================================================================
# 9. GRID SEARCH — output gap MSE criterion
# var_y_gs from normalised VAR; mn_y from unit shock MN — consistent units
# =============================================================================
print("\nRunning grid search (output gap MSE)...")
results_grid = []
var_y_gs = irfs_m_norm[:H+1, 2, 3] * 100    # normalised VAR gap in pp

for alpha in np.arange(0.30, 0.75, 0.05):
    for gamma in np.arange(0.50, 0.95, 0.05):
        for rho in np.arange(0.69, 0.92, 0.03):
            try:
                P, Q = solve_mn(gamma, alpha, rho)
                ev_max = np.max(np.abs(np.linalg.eigvals(P)))
                if ev_max >= 1.0: continue
                mn_r, mn_y, mn_pi = compute_irfs(
                    P, Q, var_r_impact_mn, H=H)    # unit shock
                if np.min(mn_r[:9]) < -0.3: continue
                if np.argmin(mn_y) < 2: continue
                score = np.mean((mn_y[:H+1] - var_y_gs[:H+1])**2)
                results_grid.append((score, alpha, gamma, rho,
                                     int(np.argmin(mn_y)),
                                     mn_y.min(), ev_max))
            except Exception:
                continue

results_grid.sort(key=lambda x: x[0])
print(f"\nTop 3:")
for s, a, g, rh, yph, yt, ev in results_grid[:3]:
    print(f"  alpha={a:.2f} gamma={g:.2f} rho={rh:.2f}  "
          f"score={s:.6f}  trough={yt:.3f}pp h={yph}")

# Update configs_A with best-fit parameters from grid search if needed
# Best fit: check top result and update if materially different from α=0.55, γ=0.90
if results_grid:
    s_b, a_b, g_b, rh_b, yph_b, yt_b, ev_b = results_grid[0]
    print(f"\nBest fit: alpha={a_b:.2f} gamma={g_b:.2f} rho={rh_b:.2f} "
          f"trough={yt_b:.3f}pp h={yph_b}")

# =============================================================================
# 10. COLOURS AND STYLES (UPDATED FOR HIGH CONTRAST)
# =============================================================================
gray_zero = '#888780'

# High-contrast, colorblind-friendly palette
COLORS = {
    'VAR':                              '#2C2C2A',    # Black
    'Base MN':                          '#D55E00',    # Vermillion Red
    'Habit IS  (α=0.55)':               '#0072B2',    # Strong Blue (Hero A)
    'Full model  (α=0.55, γ=0.90)':     '#009E73',    # Emerald Green (Full A)
    'Hybrid PC  (γ=0.50)':              '#0072B2',    # Strong Blue (Hero B)
}
STYLES = {
    'VAR':                              ('-',  2.5),
    'Base MN':                          ('--', 1.6),
    'Habit IS  (α=0.55)':               ('-',  2.0),
    'Full model  (α=0.55, γ=0.90)':     ('-',  1.8),
    'Hybrid PC  (γ=0.50)':              ('-',  2.0),
}

def make_figure(configs, irfs_dict, var_pi, var_r, var_y,
                suptitle, filename, hero_panel='y'):
    """
    Plot 1×3 figure: inflation, FFR, output gap.
    VAR in black on top. Model variants underneath.
    """
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), dpi=130)
    fig.patch.set_facecolor('white')

    panels = [
        ('Inflation (ann. pp)', 'pi', var_pi),
        ('Fed funds rate (pp)', 'r',  var_r),
        ('Output gap (pp)',     'y',  var_y),
    ]

    for ax, (title, key, var_irf) in zip(axes, panels):
        # Model lines behind VAR
        for name, gamma, alpha, rho in configs:
            ls, lw = STYLES[name]
            ax.plot(quarters, irfs_dict[name][key],
                    color=COLORS[name], ls=ls, lw=lw,
                    label=name, zorder=4)
        
        # VAR on top
        ax.plot(quarters, var_irf,
                color=COLORS['VAR'], ls='-', lw=2.5,
                label='VAR (1966–2000)', zorder=5)

        ax.axhline(0, color=gray_zero, lw=0.8, ls='--', alpha=0.6)
        
        # Apply strict, bold formatting
        ax.set_facecolor('white')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#D3D1C7')
        ax.spines['bottom'].set_color('#D3D1C7')
        
        ax.tick_params(labelsize=10, colors='black')
        ax.set_xlim(0, H)
        ax.set_xticks([0, 10, 20, 30, 40])
        ax.margins(x=0)
        
        ax.set_xlabel('Quarters', fontsize=11, fontweight='bold', color='black')
        ax.set_title(title, fontsize=12, fontweight='bold', color='black', pad=8)

    # Legend tucked safely into the hero panel's bottom right corner
    hero_col = {'y': 2, 'pi': 0}[hero_panel]
    axes[hero_col].legend(fontsize=8.5, frameon=False, loc='lower right', borderaxespad=0.5)

    # Clean, bold suptitle
    plt.suptitle(suptitle, fontsize=14, fontweight='bold',
                 color='black', y=1.04)
                 
    # Force tight spacing
    plt.tight_layout(w_pad=0.5)
    plt.savefig(filename, format='pdf', bbox_inches='tight', facecolor='white')
    print(f"Saved: {filename}")

# =============================================================================
# 11. FIGURE A — Output gap focus
# =============================================================================
make_figure(
    configs    = configs_A,
    irfs_dict  = irfs_A,
    var_pi=var_pi, var_r=var_r, var_y=var_y,
    suptitle   = '',
    filename   = 'q4_fig_A_output_gap.pdf',
    hero_panel = 'y',
)

# =============================================================================
# 12. FIGURE B — Inflation focus
# =============================================================================
make_figure(
    configs    = configs_B,
    irfs_dict  = irfs_B,
    var_pi=var_pi, var_r=var_r, var_y=var_y,
    suptitle   = '',
    filename   = 'q4_fig_B_inflation.pdf',
    hero_panel = 'y', # Set to 'y' to safely place the legend in the Output Gap panel
)

# =============================================================================
# 13. PRINT KEY NUMBERS FOR WRITEUP
# =============================================================================
print("\n=== KEY NUMBERS FOR WRITEUP ===")
print(f"\nVAR (unit shock normalised):")
print(f"  y: h=2:{var_y[2]:.3f}  h=4:{var_y[4]:.3f}  "
      f"h=8:{var_y[8]:.3f}  trough h={var_y.argmin()} val={var_y.min():.3f}pp")
print(f"  pi: h=0:{var_pi[0]:.3f}  h=1:{var_pi[1]:.3f}  h=4:{var_pi[4]:.3f}")
print(f"  r:  h=0:{var_r[0]:.3f}  h=1:{var_r[1]:.3f}")

print("\n--- Figure A models ---")
for name, gamma, alpha, rho in configs_A:
    d = irfs_A[name]
    print(f"\n{name}:")
    print(f"  y:  h=0:{d['y'][0]:.3f}  h=2:{d['y'][2]:.3f}  "
          f"h=4:{d['y'][4]:.3f}  h=8:{d['y'][8]:.3f}  "
          f"trough h={d['y'].argmin()} val={d['y'].min():.3f}pp")
    print(f"  pi: h=0:{d['pi'][0]:.3f}  h=1:{d['pi'][1]:.3f}  "
          f"h=4:{d['pi'][4]:.3f}")
    print(f"  r:  h=0:{d['r'][0]:.3f}  h=1:{d['r'][1]:.3f}")

print("\n--- Figure B models ---")
for name, gamma, alpha, rho in configs_B:
    d = irfs_B[name]
    print(f"\n{name}:")
    print(f"  y:  h=0:{d['y'][0]:.3f}  h=2:{d['y'][2]:.3f}  "
          f"trough h={d['y'].argmin()} val={d['y'].min():.3f}pp")
    print(f"  pi: h=0:{d['pi'][0]:.3f}  h=1:{d['pi'][1]:.3f}  "
          f"h=4:{d['pi'][4]:.3f}")

n̄ (sample mean, 1966–2000): -7.3463
gap_MN range: -0.1074 to 0.0921
VAR sample: 1966-01-01 to 2000-10-01, N=140
Computing bootstrap CI...
  Valid replications: 500 / 500

Unit shock normalisation:
  Raw FFR impact: 0.8577pp  norm factor: 1.1658
  FFR at h=0 after norm: 1.0000pp

VAR (unit shock): y_trough=-0.291pp at h=10
VAR FFR at h=0: 1.000pp  ← should be 1.000
=== Figure A models ===

Model                                 max|λ|    y_trough     h
  Base MN                              0.655      -0.127pp  h=0
  Habit IS  (α=0.55)                   0.733      -0.264pp  h=2
  Full model  (α=0.55, γ=0.90)         0.725      -0.260pp  h=2

=== Figure B models ===

Model                                 max|λ|    y_trough     h
  Base MN                              0.655      -0.127pp  h=0
  Hybrid PC  (γ=0.50)                  0.708      -0.111pp  h=0

Running grid search (output gap MSE)...

Top 3:
  alpha=0.55 gamma=0.90 rho=0.69  score=0.025388  trough=-0.260pp h=2
  alpha=0.55 gam